# Notebook 03 — A/B Testing & Statistical Significance
**Goal:** Statistically validate whether channel (cellular vs telephone) produces significantly different conversion rates.
Apply chi-square, z-test, and t-test to quantify campaign lift and significance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')
from ab_testing import (
    proportions_ztest, chi_square_test, ttest_by_group,
    print_ab_report, plot_conversion_comparison, plot_lift_waterfall
)
from eda_utils import save

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Records: {len(df):,}')

## 1. Define Control vs Treatment
- **Control** = Telephone (traditional channel)
- **Treatment** = Cellular (modern channel)

Hypothesis: Cellular contact produces a statistically significant higher conversion rate.

In [ ]:
control = df[df['contact'] == 'telephone']
treatment = df[df['contact'] == 'cellular']

print(f"Control   (Telephone): {len(control):,} records, {control['subscribed'].sum()} converted")
print(f"Treatment (Cellular) : {len(treatment):,} records, {treatment['subscribed'].sum()} converted")
print(f"\nControl conv rate  : {control['subscribed'].mean()*100:.2f}%")
print(f"Treatment conv rate: {treatment['subscribed'].mean()*100:.2f}%")

## 2. Two-Proportion Z-Test (Primary Test)

In [ ]:
result = proportions_ztest(
    n_A=len(control),   conv_A=control['subscribed'].sum(),
    n_B=len(treatment), conv_B=treatment['subscribed'].sum()
)
print_ab_report(result, label='Cellular vs Telephone — Conversion Rate Z-Test')

## 3. Visualize Channel Comparison

In [ ]:
fig = plot_conversion_comparison({
    'Telephone\n(Control)': control['subscribed'].mean(),
    'Cellular\n(Treatment)': treatment['subscribed'].mean()
}, title='A/B Test: Conversion Rate by Contact Channel')
save(fig, '10_ab_channel_comparison.png')
plt.show()

In [ ]:
fig = plot_lift_waterfall(control['subscribed'].mean(), treatment['subscribed'].mean())
save(fig, '11_lift_waterfall.png')
plt.show()

## 4. Chi-Square Test — Job Type vs Conversion

In [ ]:
chi_result = chi_square_test(df, 'job')
print(f"Chi-Square Statistic : {chi_result['chi2']}")
print(f"P-Value              : {chi_result['p_value']}")
print(f"Degrees of Freedom   : {chi_result['dof']}")
print(f"Significant          : {chi_result['is_significant']}")
print("\nContingency Table:")
ct = chi_result['contingency_table']
ct['conv_rate'] = (ct[1] / (ct[0]+ct[1]) * 100).round(2)
print(ct.sort_values('conv_rate', ascending=False))

## 5. Chi-Square Test — Education vs Conversion

In [ ]:
chi_edu = chi_square_test(df, 'education')
print(f"Chi-Square: {chi_edu['chi2']}, p={chi_edu['p_value']}, Significant: {chi_edu['is_significant']}")
print(chi_edu['contingency_table'])

## 6. T-Test — Age: Converters vs Non-Converters

In [ ]:
ttest = ttest_by_group(df, 'subscribed', 'age', 0, 1)
print(f"\nAge T-Test: Non-Subscribers vs Subscribers")
print(f"  Mean age (non-subscribers): {ttest['mean_a']:.1f}")
print(f"  Mean age (subscribers)    : {ttest['mean_b']:.1f}")
print(f"  T-Statistic: {ttest['t_statistic']}")
print(f"  P-Value    : {ttest['p_value']}")
print(f"  Significant: {ttest['is_significant']}")

## 7. Season-Level A/B Comparison

In [ ]:
seasons = df.groupby('season')['subscribed'].agg(['mean','count']).reset_index()
seasons.columns = ['season','conv_rate','n']
seasons['conv_rate'] = (seasons['conv_rate']*100).round(2)
seasons = seasons.sort_values('conv_rate', ascending=False)
print("Seasonal Conversion Rates:")
print(seasons.to_string(index=False))

# Pairwise chi-square: best vs worst season
best = seasons.iloc[0]['season']
worst = seasons.iloc[-1]['season']
chi_season = chi_square_test(df[df['season'].isin([best, worst])], 'season')
print(f"\nBest ({best}) vs Worst ({worst}) season:")
print(f"  Chi2={chi_season['chi2']}, p={chi_season['p_value']}, Significant={chi_season['is_significant']}")

## 8. Statistical Testing Summary

In [ ]:
summary = pd.DataFrame([
    {'Test': 'Z-Test', 'Comparison': 'Cellular vs Telephone', 'p-value': result['p_value'], 'Significant': result['is_significant'], 'Lift': f"{result['lift_pct']:+.1f}%"},
    {'Test': 'Chi-Square', 'Comparison': 'Job Type vs Conversion', 'p-value': chi_result['p_value'], 'Significant': chi_result['is_significant'], 'Lift': 'N/A'},
    {'Test': 'Chi-Square', 'Comparison': 'Education vs Conversion', 'p-value': chi_edu['p_value'], 'Significant': chi_edu['is_significant'], 'Lift': 'N/A'},
    {'Test': 'T-Test', 'Comparison': 'Age: Subscribers vs Non', 'p-value': ttest['p_value'], 'Significant': ttest['is_significant'], 'Lift': 'N/A'},
])
print(summary.to_string(index=False))